In [0]:
from pyspark.sql import functions as F

In [0]:
%sql
use catalog medallion_catalog

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS medallion_catalog.adf_test
MANAGED LOCATION 'abfss://medallion@abhiramblobstorage.dfs.core.windows.net/adfdirectory';

In [0]:

df = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "json")
         .option(
             "cloudFiles.schemaLocation",
             "/Volumes/medallion_catalog/landing/landing_volume/auto_loader/schema/"
         )
         .load("/Volumes/medallion_catalog/landing/landing_volume/auto_loader/files/")
)

In [0]:
df = df.withColumn(
    "filename",
    F.col("_metadata.file_name")
)

In [0]:
df.printSchema()

In [0]:
checkpoint_path = "/Volumes/medallion_catalog/landing/landing_volume/auto_loader/checkpoints/"

In [0]:
(df.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation", 
        checkpoint_path
    )
    .trigger(availableNow=True) 
    .table("medallion_catalog.adf_test.customers")
)

In [0]:
%sql
SELECT * FROM medallion_catalog.adf_test.customers

In [0]:
%sql
SELECT COUNT(*) FROM medallion_catalog.adf_test.customers

In [0]:
%sql
SELECT filename,Count(*) FROM medallion_catalog.adf_test.customers GROUP BY filename

In [0]:
%sql
SELECT * FROM medallion_catalog.adf_test.customers VERSION AS OF 1
EXCEPT
SELECT * FROM medallion_catalog.adf_test.customers VERSION AS OF 0


In [0]:
bronze_df = spark.sql("SELECT * FROM medallion_catalog.adf_test.customers")

In [0]:
bronze_df.show()

In [0]:
temp_df = bronze_df.drop("_rescued_data", "filename")

In [0]:
temp_df.show()


In [0]:
temp_df = temp_df.dropDuplicates(["customer_id"])

In [0]:
temp_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("medallion_catalog.adf_test.customers_silver")

In [0]:
%sql
SELECT * FROM medallion_catalog.adf_test.customers_silver

In [0]:
silver_df = spark.sql("SELECT * FROM medallion_catalog.adf_test.customers_silver")

In [0]:


gold_df = (
    silver_df
    .groupBy("city")
    .agg(
        F.sum("salary").alias("total_salary")
    )
)

In [0]:
gold_df.show()

In [0]:
gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("medallion_catalog.adf_test.customers_gold")

In [0]:
%sql
SELECT * FROM  medallion_catalog.adf_test.customers_gold